## Notebook36

### Setup

Run all of the following before starting the notebook.

In [ ]:
! wget -q -nc https://raw.githubusercontent.com/taylor-arnold/fds2/refs/heads/main/funs.py

In [ ]:
! pip install requests-cache --quiet

In [ ]:
import time

import polars as pl
from plotnine import ggplot, aes
from polars import col as c

import funs

import requests
import requests_cache

In [ ]:
session = requests_cache.CachedSession(
    cache_name="data/turing/requests_cache",
    backend="sqlite",
    allowable_methods=("GET", "HEAD", "POST"),
    allowable_codes=(200, 404),
    expire_after=None
)

base_url = "https://en.wikipedia.org/w/api.php"
headers = {
    "User-Agent": "DataScienceBook/1.0 (yourname@example.edu)"
}

### Questions

This notebook picks up the Turing Award laureate corpus notebook35 started:
the same 81 recipients of `Category:Turing Award laureates`, the same
`data/turing/requests_cache` cache file, and the same identifier pair
(`page_id` stable, `doc_id` readable). Nothing here starts a second
population; every question below attaches new data to the 81 rows already
defined. That includes rebuilding `laureates` itself in question 1 exactly as
notebook35 did, since a notebook cannot assume another notebook's Python
session is still around, only its cache file. Print results unless a
question says otherwise.

1. In the first block, rewrite `polite_get` exactly as in notebook35. In the
second block, send one `list="categorymembers"` request for `"Category:Turing
Award laureates"` on its own and print `.from_cache` for that single call. In
the third block, rewrite `category_members` exactly as in notebook35, rebuild
the full `laureates` table, and print its shape.

**Answer**:


2. In the first block, write a small `add_doc_id` helper (a join back to
`laureates`, keyed on `page_id`, with both key columns moved to the front). In
the second block, collect each laureate's outgoing internal links with
`prop="links"`, `plnamespace=0`, `pllimit="max"`, paginating with `continue`
the same way category members did, attach `doc_id` with `add_doc_id`, and print
the shape of the resulting `links` table.

**Answer**:


3. In the first block, join `links` back onto `laureates` (matching
`link_title` to `doc_id`) to keep only the edges that land on another laureate,
and print its row count. In the second block, group the result by `doc_id`,
count edges per laureate into `n_links`, and print `.describe()` for that
column. Something about the numbers should look wrong before you even check a
single row. In the third block, use `prop="templates"` on one flagged
laureate's page to find the explanation, and print the offending template's
title.

**Answer**:


4. In the first block, collect the full revision history for every laureate
(`prop="revisions"`, `rvprop="ids|timestamp|user|size|comment"`,
`rvlimit="max"`, paginating on `continue`), parse `timestamp` into a real
datetime, and print the shape. In the second block, group by `doc_id`, take the
earliest `timestamp` per laureate as that article's creation date, and print
the 5 earliest-created articles. In the third block, print the 5 most recently
created.

**Answer**:


5. In the first block, collect daily page views for every laureate from the
REST API over the same 2025 window the chapter uses (`all-access`, `user`,
`daily`, `20250101` to `20251231`), skipping any 404 the way the chapter does,
and print the shape. In the second block, sum `views` per laureate and print
the 5 highest totals.

**Answer**:


6. Plot Geoffrey Hinton's daily 2025 page views with `geom_line()`, save it,
and look at it. Hinton's total comfortably beat Tim Berners-Lee's in
question 5 despite Berners-Lee's article having a 15-year head start and
far more interwiki reach (144 language editions, from question 7, against
65 for Hinton). What does the rest of the corpus already on hand suggest
about why, and what does the plot itself leave unexplained?

**Answer**:


7. In the first block, collect `langlinks` for every laureate (batched 50 page
IDs per request, paginating with `continue`), count distinct languages per
laureate, and print the 6 laureates with the most language editions. In the
second block, print the 6 with the fewest.

**Answer**:


8. A hand-typed list of names, unlike a category listing, comes with typos,
casual capitalization, and titles that belong to someone else entirely.
Write `resolve_titles(titles)`: send one batched request with `redirects=1`,
walk the `normalized` and `redirects` maps until they stop changing (exactly
as the book does for its second corpus), and return each title's resolved
title and page ID. Run it on `"donald knuth"`, `"Edsger_W_Dijkstra"`,
`"Herbert Simon"`, `"Alan Turing"`, and `"Not A Real Laureate Page"`, then
add a column checking whether each resolved `page_id` is one of `laureates`'
own. What happened to each of the five, and why isn't "resolves to a real,
unambiguous page" the same test as "belongs to this population"?

**Answer**:


9. In the first block, rebuild `texts` exactly as notebook35 did (it should be
fast; the cache already has every request) and print its shape. In the second
block, write `laureates`, `texts`, `links`, `revisions`, `views`, and
`langlinks` to `data/turing/` as Parquet files. In the third block, write a
short manifest recording the retrieval date, the population definition, and
each table's row count, and print the manifest.

**Answer**:


10. In the first block, pull the most recent `rev_id` for `"Donald Knuth"` out
of `revisions`, then request that exact revision's wikitext with `revids=` and
`prop="revisions"`, `rvprop="ids|timestamp|content"`, `rvslots="main"`, print
the returned timestamp and the first 200 characters, and confirm the timestamp
matches the row `revisions` already had for that ID. In the second block,
resend question 1's population request with `formatversion=2` added and print
the type of `data["query"]["categorymembers"]` next to what it was without the
parameter.
Why does pinning a revision ID matter more here than pinning a format version,
even though the callout on API versioning treats them side by side?

**Answer**:


11. Between notebook35 and this one, the corpus grew from a page ID and a
title to six tables covering text, links, revisions, views, and
languages, all traceable back to one category listing. Name one place in
this corpus where that population definition (direct members of a single,
official award category, no subcategories) worked in the corpus's favor
compared to the chapter's own novelist categories, and one place a
different population would have changed an answer you printed in this
notebook.

**Answer**:
